# 3D UNet attempt

In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet3D, self).__init__()

        self.encoder1 = self.conv_block(in_channels, 64)
        self.encoder2 = self.conv_block(64, 128)
        self.encoder3 = self.conv_block(128, 256)
        self.encoder4 = self.conv_block(256, 512)
        
        # Bottleneck layer
        self.bottleneck = self.conv_block(512, 512)
        
        # Decoder layers
        self.decoder4 = self.conv_block(512 + 512, 512)  # Concatenate with enc4
        self.decoder3 = self.conv_block(256 + 512, 256)  # Concatenate with enc3
        self.decoder2 = self.conv_block(128 + 256, 128)  # Concatenate with enc2
        self.decoder1 = self.conv_block(64 + 128, 64)    # Concatenate with enc1
        
        self.final_conv = nn.Conv3d(64, out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(F.max_pool3d(enc1, kernel_size=2))
        enc3 = self.encoder3(F.max_pool3d(enc2, kernel_size=2))
        enc4 = self.encoder4(F.max_pool3d(enc3, kernel_size=2))
        
        bottleneck = self.bottleneck(F.max_pool3d(enc4, kernel_size=2))

        # Decoder with proper concatenation
        dec4 = F.interpolate(bottleneck, scale_factor=2) 
        dec4 = torch.cat((dec4, enc4), dim=1) 
        dec4 = self.decoder4(dec4)

        dec3 = F.interpolate(dec4, scale_factor=2) 
        print(dec3.shape, enc3.shape)
        dec3 = torch.cat((dec3, enc3), dim=1) 
        dec3 = self.decoder3(dec3)

        dec2 = F.interpolate(dec3, scale_factor=2) 
        dec2 = torch.cat((dec2, enc2), dim=1) 
        dec2 = self.decoder2(dec2)

        dec1 = F.interpolate(dec2, scale_factor=2) 
        dec1 = torch.cat((dec1, enc1), dim=1) 
        dec1 = self.decoder1(dec1)

        return torch.sigmoid(self.final_conv(dec1))

# Example usage:
model_3dunet_fixed = UNet3D(in_channels=1, out_channels=1).to('cuda' if torch.cuda.is_available() else 'cpu')

In [56]:
# Create a fake input tensor with correct shape [batch_size=8, channels=1, depth=100, height=100, width=100]
input_tensor = torch.rand(8, 1, 64, 64, 64)

# Forward pass
output = model_3dunet_fixed(input_tensor)
print(f"Output shape: {output.shape}")

torch.Size([8, 512, 16, 16, 16]) torch.Size([8, 256, 16, 16, 16])
Output shape: torch.Size([8, 1, 64, 64, 64])


In [23]:
volume_size=(64, 64, 64)
volume = torch.rand(volume_size)*100
volume.mean()

tensor(49.8295)

In [68]:
from torch.utils.data import Dataset, DataLoader

class Fake3DVolumeDataset(Dataset):
    def __init__(self, num_samples=50, volume_size=(128, 128, 128)):
        self.num_samples = num_samples
        self.volume_size = volume_size

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Generate random 3D volume with proper channel dimension
        if torch.rand(1) < 0.5:
            # Add channel dimension: [C, D, H, W]
            volume = torch.rand(1, *self.volume_size)  # Note the 1 channel
            label = torch.tensor([0], dtype=torch.float32)
        else:
            volume = torch.rand(1, *self.volume_size) * 100  # Note the 1 channel
            label = torch.tensor([1], dtype=torch.float32)
        return volume, label

# Instantiate the dataset
dataset = Fake3DVolumeDataset()

# Create a DataLoader
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Example of iterating through the dataloader
for batch_idx, (volumes, labels) in enumerate(dataloader):
    print(f"Batch {batch_idx}:")
    print(f"Volumes shape: {volumes.shape}")
    print(f"Labels: {labels}")

Batch 0:
Volumes shape: torch.Size([8, 1, 128, 128, 128])
Labels: tensor([[0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.]])
Batch 1:
Volumes shape: torch.Size([8, 1, 128, 128, 128])
Labels: tensor([[1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.]])
Batch 2:
Volumes shape: torch.Size([8, 1, 128, 128, 128])
Labels: tensor([[0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.]])
Batch 3:
Volumes shape: torch.Size([8, 1, 128, 128, 128])
Labels: tensor([[0.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.]])
Batch 4:
Volumes shape: torch.Size([8, 1, 128, 128, 128])
Labels: tensor([[0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.]])
Batch 5:
Volumes shape: torch.Size([8, 1, 128, 128, 128])
Labels: tensor([[1.],
        [0.],
        [1.

In [69]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_3dunet_fixed.parameters(), lr=1e-3)

In [70]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 1 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [71]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(dataloader, model_3dunet_fixed, loss_fn, optimizer)
    # test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
torch.Size([8, 512, 32, 32, 32]) torch.Size([8, 256, 32, 32, 32])


RuntimeError: Expected target size [8, 128, 128, 128], got [8, 1]

## Claude Implementation

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class Fake3DVolumeDataset(Dataset):
    def __init__(self, num_samples=50, volume_size=(128, 128, 128)):
        self.num_samples = num_samples
        self.volume_size = volume_size

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Generate random 3D volume with proper channel dimension
        if torch.rand(1) < 0.5:
            volume = torch.rand(1, *self.volume_size)  # [1, D, H, W]
            label = torch.tensor([0]).float()
        else:
            volume = torch.rand(1, *self.volume_size) * 100
            label = torch.tensor([1]).float()
        return volume, label

class UNet3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet3D, self).__init__()

        self.encoder1 = self.conv_block(in_channels, 32)
        self.encoder2 = self.conv_block(32, 64)
        self.encoder3 = self.conv_block(64, 128)
        self.encoder4 = self.conv_block(128, 256)
        
        # Bottleneck layer
        self.bottleneck = self.conv_block(256, 256)
        
        # Decoder layers
        self.decoder4 = self.conv_block(256 + 256, 256)
        self.decoder3 = self.conv_block(128 + 256, 128)
        self.decoder2 = self.conv_block(64 + 128, 64)
        self.decoder1 = self.conv_block(32 + 64, 32)
        
        # Classification head
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(32, out_channels)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(F.max_pool3d(enc1, kernel_size=2))
        enc3 = self.encoder3(F.max_pool3d(enc2, kernel_size=2))
        enc4 = self.encoder4(F.max_pool3d(enc3, kernel_size=2))
        
        bottleneck = self.bottleneck(F.max_pool3d(enc4, kernel_size=2))

        dec4 = F.interpolate(bottleneck, size=enc4.shape[2:])
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4(dec4)

        dec3 = F.interpolate(dec4, size=enc3.shape[2:])
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)

        dec2 = F.interpolate(dec3, size=enc2.shape[2:])
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)

        dec1 = F.interpolate(dec2, size=enc1.shape[2:])
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        
        # Classification head
        x = self.avg_pool(dec1)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return torch.sigmoid(x)

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet3D(in_channels=1, out_channels=1).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Create dataset and dataloader
dataset = Fake3DVolumeDataset()
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Training loop
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch_idx, (volumes, labels) in enumerate(dataloader):
        volumes = volumes.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(volumes)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 1 == 0:
            print(f'Batch {batch_idx}, Loss: {loss.item():.4f}')
    
    return total_loss / len(dataloader)

epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_epoch(model, dataloader, optimizer, criterion, device)
    # test(test_dataloader, model, loss_fn)
print("Done!")
# Run training
# print("Starting training...")
# loss = train_epoch(model, dataloader, optimizer, criterion, device)
# print(f"Epoch loss: {loss:.4f}")

Epoch 1
-------------------------------
Batch 0, Loss: 0.7334
Batch 1, Loss: 0.5154
Batch 2, Loss: 0.3030
Batch 3, Loss: 0.3124
Batch 4, Loss: 0.5369
Batch 5, Loss: 0.2406
Batch 6, Loss: 0.2347
Epoch 2
-------------------------------
Batch 0, Loss: 0.2566


KeyboardInterrupt: 

In [7]:
next(iter(dataloader))[0][0].shape

torch.Size([1, 128, 128, 128])

In [9]:
model(next(iter(dataloader))[0][0].resize(1,1,128,128,128))

/Users/arnaud/miniconda3/envs/siamese/lib/python3.11/site-packages/torch/_tensor.py:868: UserWarning: non-inplace resize is deprecated
  warnings.warn("non-inplace resize is deprecated")


tensor([[0.4857]], grad_fn=<SigmoidBackward0>)